In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests

from pyjstat import pyjstat
from collections import OrderedDict

In [2]:
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany (until 1990 former territory of the FRG)" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "GR",
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "GB",
    "Iceland" : "IS",
    "Norway" : "NO",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Turkey" : "TR",
    "Bosnia and Herzegovina" : "BA",
    "Kosovo (under United Nations Security Council Resolution 1244/99)" : "XK",
    "Moldova" : "MD",
    "Ukraine" : "UA",
    "Georgia" : "GE"
}

In [4]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder
indicator = 'nrg_cb_e'
dataformat = 'json'

params = dict(
    siec = {'E7000'},
    sinceTimePeriod = '2014',
    precision = 1,
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'GWH',
    nrg_bal = {'NEP','IMP','EXP','DL'}
)

In [ ]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)
nrg_cb_e = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

In [ ]:
#rename countries and convert to MWh
nrg_cb_e = nrg_cb_e.dropna()
nrg_cb_e = nrg_cb_e.rename(columns={'geo':'country'})
nrg_cb_e['country'] = nrg_cb_e['country'].map(map_country_ISO)
nrg_cb_e['load_MWh'] = nrg_cb_e['value'] * 1000
nrg_cb_e.head()

In [ ]:
nrg_cb_e_pivot = nrg_cb_e.pivot_table(index=['country','time'],columns='nrg_bal',values='load_MWh')
nrg_cb_e_pivot['load_MWh'] = nrg_cb_e_pivot['Net electricity production'] + nrg_cb_e_pivot['Imports'] - nrg_cb_e_pivot['Exports'] + nrg_cb_e_pivot['Distribution losses']
nrg_cb_e_pivot['DL_share'] = nrg_cb_e_pivot['Distribution losses']/nrg_cb_e_pivot['load_MWh']
nrg_cb_e_pivot.head()

In [ ]:
df_load = pd.DataFrame(nrg_cb_e_pivot['load_MWh'])

In [ ]:
df_load = pd.pivot_table(df_load, values='load_MWh', index=['country'], columns=['time'])
df_load.head()

In [ ]:
df_load.to_csv(dir_out+'load_yearly_Eurostat.csv', encoding="utf-8")

In [ ]:
nrg_cb_e_pivot['DL_share']